<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_10_1_timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 7: PyTorch Building Blocks**  

- Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
- For more information visit the [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Module 7 Material


* Part 7.1: Model Structure** [[Video]]() [[Notebook]](t81_558_class_07_1_model_structure.ipynb)
* Part 7.2: Learnable Layers [[Video]]() [[Notebook]](t81_558_class_07_2_learnable_layers.ipynb)
* Part 7.3: Nonlinearities (Activations) [[Video]]() [[Notebook]](t81_558_class_07_3_transfer.ipynb)
* **Part 7.4: Normalization & Regularization** [[Video]]() [[Notebook]](t81_558_class_07_4_normalization.ipynb)
* Part 7.5: Shape [[Video]]() [[Notebook]](t81_558_class_07_5_shapes.ipynb)



# Google CoLab Instructions

The following code checks that Google CoLab is and sets up the correct hardware settings for PyTorch.


In [ ]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 2.5)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Part 7.4: Normalization & Regularization

Normalization and regularization improve stability and generalization. Normalization layers adjust intermediate outputs to maintain consistent scale during training. Regularization techniques such as dropout reduce overfitting by limiting reliance on specific neurons. Together, these methods support reliable and efficient learning.


## Why Normalization Matters

As data flows through a deep network, the distribution of activations at each layer can shift dramatically. Early layers may produce outputs with one scale, while deeper layers receive inputs whose mean and variance drift as weights update. This phenomenon, often called internal covariate shift, slows training and forces the optimizer to constantly adapt to changing input statistics. The practical consequences include vanishing or exploding activations, sensitivity to weight initialization, and the need for very small learning rates to avoid divergence.

Normalization layers address this problem by rescaling intermediate outputs so that each layer sees inputs with a stable, well-conditioned distribution. By holding the mean and variance of activations near fixed targets, normalization decouples the learning problem at each layer from the behavior of the layers below it. The result is faster convergence, tolerance for higher learning rates, and reduced dependence on careful initialization. Normalization also has a mild regularizing effect because the per-batch statistics inject a small amount of noise into the forward pass.

## Batch Normalization

Batch normalization is the most widely used normalization technique for fully connected and convolutional networks. For each feature, it computes the mean and variance across the current mini-batch, subtracts the mean, and divides by the standard deviation. It then applies a learned scale and shift, which allows the network to recover any distribution it actually needs rather than being locked into a strict zero-mean, unit-variance form.

During training, batch normalization uses statistics from the current mini-batch. During evaluation, it switches to running averages of the mean and variance accumulated during training, so predictions do not depend on the composition of any particular batch. This is one reason it is essential to call `model.eval()` before scoring or generating predictions: it tells PyTorch to use the stored running statistics rather than recomputing them from a possibly small or unrepresentative evaluation batch.

PyTorch provides batch normalization through `nn.BatchNorm1d` for tabular or sequence features, `nn.BatchNorm2d` for image feature maps, and `nn.BatchNorm3d` for volumetric data. The most common practice is to insert batch normalization between a linear or convolutional layer and its activation function.

## Layer Normalization and Other Variants

Batch normalization works well when batches are large and representative, but it struggles in settings where batch size is small, where sequence lengths vary, or where samples within a batch differ in fundamental ways. Layer normalization (`nn.LayerNorm`) addresses these cases by normalizing across the features of a single sample rather than across the batch dimension. Because each sample is normalized independently, batch size has no effect on the computed statistics, and training and evaluation behave identically. Layer normalization is the default choice in transformer architectures for this reason.

Other variants exist for specific situations. Group normalization splits channels into groups and normalizes within each group, providing a middle ground between layer and instance normalization that performs well in computer vision when batch sizes are constrained. Instance normalization normalizes each sample and each channel independently and is common in style transfer. Choosing among these variants typically comes down to the architecture and the practical batch size available during training.

## Regularization and the Overfitting Problem

A neural network with sufficient capacity can memorize its training data, achieving very low training loss while performing poorly on unseen examples. Regularization is the family of techniques that discourages this kind of memorization and pushes the model toward solutions that generalize. The goal is not to make training loss as low as possible but to make the gap between training and validation performance as small as possible while keeping both acceptably low.

Several forms of regularization are common in practice. Weight decay, implemented in PyTorch through the `weight_decay` argument of the optimizer, penalizes the squared magnitude of the weights and discourages the network from relying on any single large parameter. Early stopping monitors validation loss and halts training when it stops improving, preventing the optimizer from continuing to specialize on training noise. Data augmentation expands the effective size of the training set by applying random transformations that preserve the label. Each of these techniques makes the network's job harder in a controlled way, which forces it to learn features that survive perturbation.

## Dropout

Dropout is one of the most effective and widely used regularization techniques in deep learning. During training, a dropout layer randomly sets a fraction of its input activations to zero on each forward pass. The remaining activations are scaled up so that the expected sum of the outputs matches the input, which keeps the magnitude of downstream computations stable. The fraction of units dropped is controlled by a probability `p`, with values between 0.2 and 0.5 being typical for hidden layers.

The key insight behind dropout is that it prevents neurons from forming brittle co-adaptations. If any individual neuron might be removed at any moment, the network cannot rely on a specific neighbor always being available to compensate for its mistakes. Each neuron must therefore learn features that are useful on their own, and the network as a whole behaves like an ensemble of many smaller networks that share weights. At evaluation time dropout is disabled and all neurons participate, which is conceptually similar to averaging the predictions of that ensemble.

PyTorch implements dropout through `nn.Dropout` for fully connected layers and `nn.Dropout2d` for convolutional feature maps. As with batch normalization, the layer behaves differently during training and evaluation, which is another reason that calling `model.train()` and `model.eval()` at the appropriate times is essential.

## Combining Normalization and Regularization

Normalization and dropout solve different problems and are usually used together. A common pattern in fully connected networks is to follow each linear layer with batch normalization, then an activation function, then dropout. Normalization stabilizes the scale of the activations entering the nonlinearity, the activation provides representational capacity, and dropout adds the noise that discourages co-adaptation. The output layer is typically left without dropout or normalization so that the final logits or predictions are not perturbed.

When using both techniques, a few practical points are worth keeping in mind. Batch normalization already provides a small amount of stochastic regularization through its batch statistics, so very aggressive dropout on top of it is rarely necessary. Layer normalization and dropout combine cleanly because layer normalization does not introduce batch-dependent noise. Weight decay can be combined with either technique, though excessive weight decay alongside heavy dropout can underfit the model. As always, the right combination depends on the dataset and architecture, and validation performance is the final arbiter.

## Python Example

The following example builds a small fully connected network that combines linear layers, batch normalization, ReLU activations, and dropout. The network is trained on a synthetic binary classification problem so the focus stays on the structure rather than on the dataset. Notice the explicit calls to `model.train()` and `model.eval()`, which switch both the batch normalization and dropout layers between their training and evaluation behaviors.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Generate a synthetic binary classification dataset.
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train, dtype=torch.long, device=device)
X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
y_test_t = torch.tensor(y_test, dtype=torch.long, device=device)

# Define a network that uses batch normalization and dropout.
class RegularizedNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_classes=2, dropout_p=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, num_classes)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_p)

    def forward(self, x):
        x = self.dropout(self.act(self.bn1(self.fc1(x))))
        x = self.dropout(self.act(self.bn2(self.fc2(x))))
        return self.fc_out(x)

model = RegularizedNet(input_dim=X_train.shape[1]).to(device)

# Weight decay in the optimizer adds L2 regularization on top of dropout.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Simple training loop with explicit train/eval mode switching.
num_epochs = 50
batch_size = 64

for epoch in range(num_epochs):
    model.train()
    perm = torch.randperm(X_train_t.size(0))
    epoch_loss = 0.0
    for i in range(0, X_train_t.size(0), batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = X_train_t[idx], y_train_t[idx]
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= X_train_t.size(0)

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_logits = model(X_test_t)
            test_pred = test_logits.argmax(dim=1)
            test_acc = (test_pred == y_test_t).float().mean().item()
        print(f"Epoch {epoch + 1:3d} | train loss {epoch_loss:.4f} | test acc {test_acc:.4f}")

# Final evaluation.
model.eval()
with torch.no_grad():
    final_pred = model(X_test_t).argmax(dim=1)
    final_acc = (final_pred == y_test_t).float().mean().item()
print(f"Final test accuracy: {final_acc:.4f}")